# Two-round optical-flow alignment QC validation

This notebook runs the same normalization, Farnebäck flow, warping, residual, coordinate-conversion, and cell-neighborhood aggregation functions used by `mif-pipeline alignment-qc`. It is intentionally read-only: it opens an existing canonical SpatialData store but does not write pipeline artifacts or modify the store.

Choose two exact aliases already present in `full_image`. Alias text is not interpreted, so you control whether the selected channels are imaging or AF acquisitions.

## Environment

Run this notebook in the same modern SpatialData environment used for `assemble-spatialdata` and install the optional OpenCV dependency with `pip install -e '.[alignment-qc]'`. The selected pyramid-level images and dense fields are materialized in memory.

In [ ]:
from pathlib import Path
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from spatialdata import read_zarr

from mif_pipeline import load_config
from mif_pipeline.config import get_slide_config, resolve_channel_entries
from mif_pipeline.alignment_qc import (
    DEFAULT_FLOW_PARAMS,
    DENSE_METRIC_NAMES,
    _cell_observations,
    _channel_names,
    _image_levels,
    _materialize_channel,
    _round_summary,
    compute_flow_residual_maps,
    local_dapi_support,
    neighborhood_radii_pixels,
    normalize_percentile_image,
    sample_neighborhood_nanmedian,
    select_pyramid_level,
)

plt.rcParams['figure.dpi'] = 120

## Select the slide and two aliases

Set `PYRAMID_LEVEL` to an integer and `TARGET_RESOLUTION_UM` to `None` to force a specific stored level. Otherwise, the production selector chooses the stored level nearest the target resolution. A `2.6 µm` sampling radius produces a 3×3 window when the selected level is `2.6 µm/pixel`.

In [ ]:
CONFIG_PATH = Path('../example.yaml')
SLIDE_ID = 'SLIDE-0272'
REFERENCE_ALIAS = 'R1_DAPI'
MOVING_ALIAS = 'R2_DAPI'

TARGET_RESOLUTION_UM = 2.6
PYRAMID_LEVEL = None
LOWER_PERCENTILE = 1.0
UPPER_PERCENTILE = 99.9
SSIM_WINDOW_SIZE = 11
CELL_SAMPLING_RADIUS_UM = 2.6
FLOW_PARAMS = dict(DEFAULT_FLOW_PARAMS)

# Plotting is decimated only for display; calculations use the complete selected level.
MAX_PLOT_DIMENSION = 2500
CELL_MARKER_SIZE = 2

In [ ]:
config = load_config(CONFIG_PATH)
slide = get_slide_config(config, SLIDE_ID)

# This validates only exact alias resolution through the existing channel map.
resolve_channel_entries(config, SLIDE_ID, [REFERENCE_ALIAS, MOVING_ALIAS])

store_path = Path(slide['spatialdata']['store_path'])
if not store_path.exists():
    raise FileNotFoundError(f'Canonical SpatialData store not found: {store_path}')

sdata = read_zarr(store_path)
if 'full_image' not in sdata.images:
    raise KeyError("Canonical store is missing images['full_image']")
if 'agg_cell_labels' not in sdata.tables:
    raise KeyError("Canonical store is missing tables['agg_cell_labels']")

levels = _image_levels(sdata.images['full_image'])
available_aliases = _channel_names(levels[0][1])
missing = [alias for alias in (REFERENCE_ALIAS, MOVING_ALIAS) if alias not in available_aliases]
if missing:
    raise KeyError(f'Aliases absent from full_image: {missing}')

print(f'SpatialData store: {store_path}')
print(f'Available pyramid levels: {[name for name, _ in levels]}')
print(f'Available aliases ({len(available_aliases)}): {available_aliases}')

In [ ]:
selected = select_pyramid_level(
    levels,
    native_pixel_size_um=float(slide['pixel_size_um']),
    pyramid_level=PYRAMID_LEVEL,
    target_resolution_um=TARGET_RESOLUTION_UM,
)
level_array = selected['array']
pixel_size_x_um = float(selected['pixel_size_x_um'])
pixel_size_y_um = float(selected['pixel_size_y_um'])
radius_x, radius_y = neighborhood_radii_pixels(
    CELL_SAMPLING_RADIUS_UM,
    pixel_size_x_um=pixel_size_x_um,
    pixel_size_y_um=pixel_size_y_um,
)

level_details = {key: value for key, value in selected.items() if key != 'array'}
level_details['sampling_radius_pixels'] = {'x': radius_x, 'y': radius_y}
level_details['sampling_window_pixels'] = {
    'x': 2 * radius_x + 1,
    'y': 2 * radius_y + 1,
}
pd.Series(level_details)

In [ ]:
reference_raw = _materialize_channel(level_array, REFERENCE_ALIAS)
moving_raw = _materialize_channel(level_array, MOVING_ALIAS)

reference_normalized, reference_normalization = normalize_percentile_image(
    reference_raw,
    lower_percentile=LOWER_PERCENTILE,
    upper_percentile=UPPER_PERCENTILE,
)
moving_normalized, moving_normalization = normalize_percentile_image(
    moving_raw,
    lower_percentile=LOWER_PERCENTILE,
    upper_percentile=UPPER_PERCENTILE,
)

pd.DataFrame(
    [reference_normalization, moving_normalization],
    index=[REFERENCE_ALIAS, MOVING_ALIAS],
)

In [ ]:
plot_step = max(1, math.ceil(max(reference_raw.shape) / MAX_PLOT_DIMENSION))
plot_slice = np.s_[::plot_step, ::plot_step]
extent_um = [
    0,
    reference_raw.shape[1] * pixel_size_x_um,
    reference_raw.shape[0] * pixel_size_y_um,
    0,
]

fig, axes = plt.subplots(2, 2, figsize=(13, 11), constrained_layout=True)
panels = [
    (reference_raw, f'{REFERENCE_ALIAS} raw'),
    (moving_raw, f'{MOVING_ALIAS} raw'),
    (reference_normalized, f'{REFERENCE_ALIAS} normalized'),
    (moving_normalized, f'{MOVING_ALIAS} normalized'),
]
for ax, (image, title) in zip(axes.flat, panels):
    shown = ax.imshow(image[plot_slice], cmap='gray', extent=extent_um, origin='upper')
    ax.set_title(title)
    ax.set_xlabel('x (µm)')
    ax.set_ylabel('y (µm)')
    fig.colorbar(shown, ax=ax, shrink=0.75)
plt.show()

In [ ]:
flow_result = compute_flow_residual_maps(
    reference_normalized,
    moving_normalized,
    pixel_size_x_um=pixel_size_x_um,
    pixel_size_y_um=pixel_size_y_um,
    flow_params=FLOW_PARAMS,
    ssim_window_size=SSIM_WINDOW_SIZE,
)
dense_maps = {name: flow_result[name] for name in DENSE_METRIC_NAMES}

print(f"Flow direction: {flow_result['flow_direction']}")
print(f"Valid dense fraction: {np.mean(flow_result['valid_mask']):.3f}")

In [ ]:
def symmetric_limit(values, percentile=99):
    finite = np.asarray(values)[np.isfinite(values)]
    return 1.0 if not len(finite) else max(float(np.percentile(np.abs(finite), percentile)), 1e-6)

def upper_limit(values, percentile=99):
    finite = np.asarray(values)[np.isfinite(values)]
    return 1.0 if not len(finite) else max(float(np.percentile(finite, percentile)), 1e-6)

fig, axes = plt.subplots(2, 3, figsize=(18, 11), constrained_layout=True)
flow_x_limit = symmetric_limit(dense_maps['flow_x_um'])
flow_y_limit = symmetric_limit(dense_maps['flow_y_um'])
dense_panels = [
    ('flow_x_um', 'Signed x displacement (µm)', 'coolwarm', -flow_x_limit, flow_x_limit),
    ('flow_y_um', 'Signed y displacement (µm)', 'coolwarm', -flow_y_limit, flow_y_limit),
    ('displacement_um', 'Displacement magnitude (µm)', 'magma', 0, upper_limit(dense_maps['displacement_um'])),
    ('warped_moving', 'Warped moving, normalized', 'gray', 0, 1),
    ('absolute_residual', 'Absolute normalized residual', 'inferno', 0, upper_limit(dense_maps['absolute_residual'])),
    ('structural_residual', 'Structural residual (1 - SSIM)', 'inferno', 0, upper_limit(dense_maps['structural_residual'])),
]
for ax, (name, title, cmap, vmin, vmax) in zip(axes.flat, dense_panels):
    image = flow_result[name]
    shown = ax.imshow(
        image[plot_slice],
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        extent=extent_um,
        origin='upper',
    )
    ax.set_title(title)
    ax.set_xlabel('x (µm)')
    ax.set_ylabel('y (µm)')
    fig.colorbar(shown, ax=ax, shrink=0.75)
plt.show()

In [ ]:
# A sparse vector overlay makes the signed direction easier to inspect.
quiver_step = max(1, math.ceil(max(reference_raw.shape) / 60))
yy, xx = np.mgrid[0:reference_raw.shape[0]:quiver_step, 0:reference_raw.shape[1]:quiver_step]
u = dense_maps['flow_x_um'][::quiver_step, ::quiver_step]
v = dense_maps['flow_y_um'][::quiver_step, ::quiver_step]

fig, ax = plt.subplots(figsize=(12, 10), constrained_layout=True)
ax.imshow(reference_normalized[plot_slice], cmap='gray', extent=extent_um, origin='upper')
ax.quiver(
    xx * pixel_size_x_um,
    yy * pixel_size_y_um,
    u,
    v,
    dense_maps['displacement_um'][::quiver_step, ::quiver_step],
    cmap='magma',
    angles='xy',
    scale_units='xy',
    scale=1,
    width=0.002,
)
ax.set_title(f'Reference-to-moving flow: {REFERENCE_ALIAS} → {MOVING_ALIAS}')
ax.set_xlabel('x (µm)')
ax.set_ylabel('y (µm)')
plt.show()

## Aggregate dense measurements at reference cell centers

The production stage obtains cell IDs and micron-space centers from `agg_cell_labels`, projects them onto the selected flow grid, and takes the neighborhood `nanmedian`. DAPI support is the unnormalized local moving/reference intensity ratio over the same neighborhood.

In [ ]:
source_obs, instance_ids, spatial_um = _cell_observations(sdata.tables['agg_cell_labels'])
x_level = spatial_um[:, 0] / pixel_size_x_um
y_level = spatial_um[:, 1] / pixel_size_y_um

cell_metrics = {
    name: sample_neighborhood_nanmedian(
        dense_maps[name],
        x_level,
        y_level,
        radius_x=radius_x,
        radius_y=radius_y,
    )
    for name in DENSE_METRIC_NAMES
}
cell_metrics['dapi_support'] = local_dapi_support(
    reference_raw,
    moving_raw,
    x_level,
    y_level,
    radius_x=radius_x,
    radius_y=radius_y,
    reference_dynamic_range=reference_normalization['normalization_dynamic_range'],
)

cell_qc = pd.DataFrame(
    {
        'instance_id': instance_ids,
        'x_um': spatial_um[:, 0],
        'y_um': spatial_um[:, 1],
        **cell_metrics,
    }
).set_index('instance_id')
cell_qc.head()

In [ ]:
cell_qc.describe(percentiles=[0.05, 0.5, 0.95]).T

In [ ]:
cell_panels = [
    ('flow_x_um', 'Cell flow x (µm)', 'coolwarm', True),
    ('flow_y_um', 'Cell flow y (µm)', 'coolwarm', True),
    ('displacement_um', 'Cell displacement (µm)', 'magma', False),
    ('absolute_residual', 'Cell absolute residual', 'inferno', False),
    ('structural_residual', 'Cell structural residual', 'inferno', False),
    ('dapi_support', 'Cell DAPI support ratio', 'viridis', False),
]
fig, axes = plt.subplots(2, 3, figsize=(18, 11), constrained_layout=True)
for ax, (name, title, cmap, diverging) in zip(axes.flat, cell_panels):
    values = cell_qc[name].to_numpy(dtype=float)
    finite = values[np.isfinite(values)]
    if diverging:
        limit = 1.0 if not len(finite) else max(np.percentile(np.abs(finite), 99), 1e-6)
        vmin, vmax = -limit, limit
    else:
        vmin = 0 if name != 'dapi_support' else (0 if not len(finite) else np.percentile(finite, 1))
        vmax = 1 if not len(finite) else max(np.percentile(finite, 99), vmin + 1e-6)
    ax.imshow(
        reference_normalized[plot_slice],
        cmap='gray',
        extent=extent_um,
        origin='upper',
        alpha=0.35,
    )
    shown = ax.scatter(
        cell_qc['x_um'],
        cell_qc['y_um'],
        c=values,
        s=CELL_MARKER_SIZE,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        linewidths=0,
    )
    ax.set_title(title)
    ax.set_xlabel('x (µm)')
    ax.set_ylabel('y (µm)')
    ax.set_aspect('equal')
    fig.colorbar(shown, ax=ax, shrink=0.75)
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9), constrained_layout=True)
for ax, name in zip(axes.flat, cell_metrics):
    values = cell_qc[name].to_numpy(dtype=float)
    values = values[np.isfinite(values)]
    if len(values):
        lower, upper = np.percentile(values, [0.5, 99.5])
        shown = values[(values >= lower) & (values <= upper)]
        ax.hist(shown, bins=80, color='steelblue', alpha=0.85)
    ax.set_title(name)
    ax.set_ylabel('cells')
plt.show()

In [ ]:
# This is the same compact summary constructor used by the production stage.
summary = _round_summary(
    alias=MOVING_ALIAS,
    index=1,
    is_reference=False,
    normalization=moving_normalization,
    maps=dense_maps,
    support=cell_metrics['dapi_support'],
)
pd.Series(summary, name=MOVING_ALIAS).to_frame()

## Interpretation notes

- Flow is stored in the reference-to-moving direction. The moving image is sampled at `reference coordinate + flow` to warp it into reference coordinates.
- High displacement alone does not prove valid correspondence; inspect residual and DAPI-support maps together.
- Structural residual is `1 - local SSIM`, a general unexplained-disagreement measure rather than a tissue-loss probability.
- DAPI support uses unnormalized intensities so a globally weak acquisition cannot be rescued solely by independent percentile normalization.
- This notebook makes no filtering decisions and writes nothing. After choosing production settings, run the explicit `alignment-qc` stage separately.